# Desenvolvimento do modelo de Multi Layer Perceptron (MLP) Classifier Binário de três camadas ocultas
---

***A Fazeres:***
- Buscar `GridSearch`
- Buscar `Ensamble`
- Buscar `LaTeX`
- Buscar `overleaf.com`

Esse notebook mostra o desenvolvimento, de maneira explicada, de um `MLP` de três camadas. Trará aplicações de `RFE`, limiares relativos à `Sensibilidade`, `Precisão` e `Equilíbrio`.

Serão teinados quatro modelos com arquitetura em funil:

1. O primeiro terá arquitetura em funil decrescente: a primeira camada oculta é a maior e, após ela, será divido o valor de neurônios pela metade) **SEM** a aplicação do `RFE`;
2. O segundo terá arquitetura em funil crescente: a primeira camada oculta é a menor e, após ela, será multiplicado por 2 o valor de neurônios, **SEM** a aplicação do `RFE`;
3. O terceiro será como o primeiro (decrescente), mas **COM** a aplicação do `RFE`;
4. O quarto será como o segundo (crescente), mas **COM** a aplicação do `RFE`;    

## A) Importando as Bibliotecas e Pacotes a serem usados
---

### 1. Pacotes baixados
---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import confusion_matrix, classification_report, RocCurveDisplay
from sklearn.feature_selection import RFE
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

### 2. Pacotes Predexan
---

In [2]:
import sys
import os

caminho_predexan = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if caminho_predexan not in sys.path:
    sys.path.append(caminho_predexan)

print(f"Pastas encontradas: {os.listdir(caminho_predexan)}")

import predexan.graficos as pg
import predexan.metricas as pm
import predexan.preproc as pp

Pastas encontradas: ['.ipynb_checkpoints', 'analise_exploratoria_pre_processamento', 'dataset', 'imagens', 'modelos', 'predexan', 'README.md']


## B) Determinando os Dataframes à partir das bases de dados pré-processados
---

### 1. Determinando df_treino
---

In [3]:
df_treino = pd.read_parquet('../../dataset/pre_processado/treino_pre_processado.parquet')
df_treino.info()

<class 'pandas.DataFrame'>
Index: 96822 entries, 0 to 175337
Data columns (total 59 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   duracao_conexao_seg                 96822 non-null  float64
 1   num_pacotes_origem                  96822 non-null  float64
 2   num_pacotes_destino                 96822 non-null  float64
 3   total_bytes_origem                  96822 non-null  float64
 4   total_bytes_destino                 96822 non-null  float64
 5   taxa_pacote                         96822 non-null  float64
 6   carga_origem                        96822 non-null  float64
 7   carga_destino                       96822 non-null  float64
 8   pacotes_perdidos_origem             96822 non-null  float64
 9   pacotes_perdidos_destino            96822 non-null  float64
 10  tempo_medio_pacote_origem           96822 non-null  float64
 11  bytes_por_pacote_destino            96822 non-null  floa

### 2. Determinando df_teste
---

In [4]:
df_teste = pd.read_parquet('../../dataset/pre_processado/teste_pre_processado.parquet')
df_teste.info()

<class 'pandas.DataFrame'>
Index: 49971 entries, 0 to 82328
Data columns (total 57 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   duracao_conexao_seg                 49971 non-null  float64
 1   num_pacotes_origem                  49971 non-null  float64
 2   num_pacotes_destino                 49971 non-null  float64
 3   total_bytes_origem                  49971 non-null  float64
 4   total_bytes_destino                 49971 non-null  float64
 5   taxa_pacote                         49971 non-null  float64
 6   carga_origem                        49971 non-null  float64
 7   carga_destino                       49971 non-null  float64
 8   pacotes_perdidos_origem             49971 non-null  float64
 9   pacotes_perdidos_destino            49971 non-null  float64
 10  tempo_medio_pacote_origem           49971 non-null  float64
 11  bytes_por_pacote_destino            49971 non-null  float

## C) Separando os valores em teste e treino
---

Nessa seção, serão separadas as `features` do `target`, tanto para a base de testes quanto de treino.

Há duas possibilidades claras de `target`, presentes nos datasets, dado o cenário de `Classificação` e `Regressão Logística`: `label`e `categoria_ataque.

A primeira retornaria um diagnóstico apontado se há uma probabilidade alta ou baixa de haver um ataque, de maneira geral. Já a segunda, retornaria a probabilidade de haver um ataque específico dentro de suas dez classes.

A escolha do target se embasa tanto na especialização da resposta, quanto no desempenho e custo computacional do modelo. Dessa forma, ao escolher `label`, a resposta é mais simples, mas continua sendo de grande valor, e o custo computacional é muito inferior. Já a escolha de `categoria_ataque`, traz uma resposta mais especializada e de maior qualidade para o usuário, contudo o custo computacional se mostrou alto.

### 1. Atributos do df_treino
---

In [5]:
# Features
X_treino = df_treino.drop(columns=['label_0', 
                                   'label_1',
                                   'categoria_ataque_Analysis',
                                   'categoria_ataque_Backdoor',
                                   'categoria_ataque_DoS',
                                   'categoria_ataque_Exploits',
                                   'categoria_ataque_Fuzzers',
                                   'categoria_ataque_Generic',
                                   'categoria_ataque_Normal',
                                   'categoria_ataque_Reconnaissance',
                                   'categoria_ataque_Shellcode',
                                   'categoria_ataque_Worms'
                                  ])
print(f'O tipo de X_treino é {type(X_treino)}. Possui {X_treino.shape[1]} features e {X_treino.shape[0]} registros.')
# Target
y_treino = df_treino['label_1'] # Escolhendo somente uma, no caso a de ataque, se torna classificação binária
print(f'O tipo de y_treino é {type(y_treino)}')

O tipo de X_treino é <class 'pandas.DataFrame'>. Possui 47 features e 96822 registros.
O tipo de y_treino é <class 'pandas.Series'>


### 2. Atributos do df_teste
---

In [6]:
# Features
X_teste = df_teste.drop(columns=['label_0', 
                                   'label_1',
                                   'categoria_ataque_Analysis',
                                   'categoria_ataque_Backdoor',
                                   'categoria_ataque_DoS',
                                   'categoria_ataque_Exploits',
                                   'categoria_ataque_Fuzzers',
                                   'categoria_ataque_Generic',
                                   'categoria_ataque_Normal',
                                   'categoria_ataque_Reconnaissance',
                                   'categoria_ataque_Shellcode',
                                   'categoria_ataque_Worms'
                                  ])
print(f'O tipo de X_teste é {type(X_teste)}. Possui {X_teste.shape[1]} features e {X_teste.shape[0]} registros.')
# Target
y_teste = df_teste['label_1'] # Escolhendo somente uma, no caso a de ataque, se torna classificação binária
print(f'O tipo de y_teste é {type(y_teste)}')

O tipo de X_teste é <class 'pandas.DataFrame'>. Possui 45 features e 49971 registros.
O tipo de y_teste é <class 'pandas.Series'>


O dataset de teste possui duas features a menos: `servico_conexao_pop3` e `servico_conexao_ssh`. Desta forma, é necessário adicionar ambas, com os valores iguais a `0.0`:

In [7]:
X_teste = X_teste.reindex(columns=X_treino.columns, fill_value=0.0)

print(f'Agora, o tipo de X_teste é {type(X_teste)}. Possui {X_teste.shape[1]} features e {X_teste.shape[0]} registros.')

Agora, o tipo de X_teste é <class 'pandas.DataFrame'>. Possui 47 features e 49971 registros.


### 3. Aplicação do Recursive Feature Elimination (RFE) para diminuição da dimensionalidade
---

O RFE usa o treino de um modelo para verificar as features mais importantes para o treino e teste. No caso, será usado XGBoost.

#### 3.1. Instanciamento da classe RFE
---

O intuito é que a dimensionalidade seja diminuida, e 10 features sejam escolhidas

In [8]:
xgb_estimador = XGBClassifier(random_state=42, eval_metric='logloss')

In [9]:
seletor_rfe = RFE(estimator=xgb_estimador, n_features_to_select=15)

#### 3.2. Aplicação do RFE em X_treino
---

In [10]:
seletor_rfe.fit_transform(X_treino, y_treino)

array([[-3.29991646e-01, -1.54433353e-01, -2.41770564e-03, ...,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-1.31161236e-01,  1.21303582e+01,  8.08271323e-01, ...,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-2.85714286e-01,  3.66647093e+00,  8.34429779e-02, ...,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       ...,
       [-4.42773601e-02,  4.69348209e+00,  1.95689892e-01, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 4.82581454e+00,  3.68761010e-01, -6.30313652e-04, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-1.78780284e-01, -1.00998238e-01, -8.29941346e-03, ...,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00]],
      shape=(96822, 15))

In [11]:
X_treino_reduzido = seletor_rfe.transform(X_treino)

In [12]:
colunas_originais = X_treino.columns
print(f'Registros e features de treino ANTES da aplicação do RFE:\n - Número de features e registros: {X_treino.shape}\n - Nomes das features: {X_treino.columns}\n')
colunas_selecionadas = colunas_originais[seletor_rfe.support_]
print(f'\nRegistros e features de treino DEPOIS da aplicação do RFE:\n - Número de features e registros:  {X_treino_reduzido.shape} \n - Nomes das features: {list(colunas_selecionadas)}')

Registros e features de treino ANTES da aplicação do RFE:
 - Número de features e registros: (96822, 47)
 - Nomes das features: Index(['duracao_conexao_seg', 'num_pacotes_origem', 'num_pacotes_destino',
       'total_bytes_origem', 'total_bytes_destino', 'taxa_pacote',
       'carga_origem', 'carga_destino', 'pacotes_perdidos_origem',
       'pacotes_perdidos_destino', 'tempo_medio_pacote_origem',
       'bytes_por_pacote_destino', 'var_lat_pacotes_origem',
       'var_lat_pacotes_destino', 'tamanho_janela_tcp_origem',
       'num_sequencia_tcp_origem', 'num_sequencia_tcp_destino',
       'tamanho_janela_tcp_destino', 'rtt_handshake_tcp', 'tempo_syn_synack',
       'tempo_synack_ack', 'tamanho_medio_pacotes_origem',
       'tamanho_medio_pacotes_destino', 'profundidade_transacao_http',
       'tamanho_corpo_resposta_http', 'conexoes_recentes_ip_porta_destino',
       'conexoes_recentes_ip_porta_origem', 'ftp_autenticada',
       'num_comandos_ftp', 'num_metodos_http',
       'ip_porta_

In [13]:
X_treino_reduzido_df = pd.DataFrame(X_treino_reduzido, columns=colunas_selecionadas)
X_treino_reduzido_df.head()

,total_bytes_origem,total_bytes_destino,carga_destino,pacotes_perdidos_origem,pacotes_perdidos_destino,rtt_handshake_tcp,tempo_syn_synack,tempo_synack_ack,tamanho_medio_pacotes_origem,tamanho_medio_pacotes_destino,protocolo_rede_outros,protocolo_rede_udp,servico_conexao_-,servico_conexao_ftp-data,estado_conexao_CON
0,-0.329992,-0.154433,-0.002418,-0.4,-0.333333,-0.009828,-0.016261,-0.004038,-0.547945,-0.274074,0.0,0.0,1.0,0.0,0.0
1,-0.131161,12.130358,0.808271,0.0,2.500000,-0.009828,-0.016261,-0.004038,-0.424658,7.600000,0.0,0.0,1.0,0.0,0.0
2,-0.285714,3.666471,0.083443,-0.2,0.666667,0.845333,0.898078,0.820533,-0.506849,5.511111,0.0,0.0,1.0,0.0,0.0
3,-0.175439,0.021139,-0.010829,-0.2,0.166667,-0.009828,-0.016261,-0.004038,-0.424658,-0.118519,0.0,0.0,0.0,0.0,0.0
4,-0.214703,-0.126248,-0.009800,0.0,-0.166667,0.971310,1.042226,0.931617,-0.410959,-0.259259,0.0,0.0,1.0,0.0,0.0


In [14]:
for feature in X_treino_reduzido_df.columns:
    print(f'\nFeature {feature.upper()}:')
    print(f'\n - Amplitude: de {X_treino_reduzido_df[feature].min()} a {X_treino_reduzido_df[feature].max()}')
    print(f'\n - Magnitude: {X_treino_reduzido_df[feature].max() - X_treino_reduzido_df[feature].min()}\n')


Feature TOTAL_BYTES_ORIGEM:

 - Amplitude: de -0.42606516290726815 a 5415.2819548872185

 - Magnitude: 5415.708020050126


Feature TOTAL_BYTES_DESTINO:

 - Amplitude: de -0.20493247210804463 a 4302.6576629477395

 - Magnitude: 4302.862595419848


Feature CARGA_DESTINO:

 - Amplitude: de -0.016328903360192524 a 36.700989512499405

 - Magnitude: 36.717318415859594


Feature PACOTES_PERDIDOS_ORIGEM:

 - Amplitude: de -0.4 a 960.2

 - Magnitude: 960.6


Feature PACOTES_PERDIDOS_DESTINO:

 - Amplitude: de -0.3333333333333333 a 913.6666666666666

 - Magnitude: 914.0


Feature RTT_HANDSHAKE_TCP:

 - Amplitude: de -0.009828122445103204 a 19.24055182089483

 - Magnitude: 19.250379943339933


Feature TEMPO_SYN_SYNACK:

 - Amplitude: de -0.016261070130149765 a 31.23165475574871

 - Magnitude: 31.24791582587886


Feature TEMPO_SYNACK_ACK:

 - Amplitude: de -0.00403792688985175 a 24.859196170876466

 - Magnitude: 24.863234097766316


Feature TAMANHO_MEDIO_PACOTES_ORIGEM:

 - Amplitude: de -0.75342

#### 3.3. Aplicação do RFE em X_teste
---

In [15]:
X_teste_reduzido = seletor_rfe.transform(X_teste)

In [16]:
colunas_originais = X_teste.columns
print(f'Registros e features de treino ANTES da aplicação do PCA:\n - Número de features e registros: {X_teste.shape}\n - Nomes das features: {X_teste.columns}\n')
colunas_selecionadas = colunas_originais[seletor_rfe.support_]
print(f'\nRegistros e features de treino DEPOIS da aplicação do PCA:\n - Número de features e registros:  {X_teste_reduzido.shape} \n - Nomes das features: {list(colunas_selecionadas)}')

Registros e features de treino ANTES da aplicação do PCA:
 - Número de features e registros: (49971, 47)
 - Nomes das features: Index(['duracao_conexao_seg', 'num_pacotes_origem', 'num_pacotes_destino',
       'total_bytes_origem', 'total_bytes_destino', 'taxa_pacote',
       'carga_origem', 'carga_destino', 'pacotes_perdidos_origem',
       'pacotes_perdidos_destino', 'tempo_medio_pacote_origem',
       'bytes_por_pacote_destino', 'var_lat_pacotes_origem',
       'var_lat_pacotes_destino', 'tamanho_janela_tcp_origem',
       'num_sequencia_tcp_origem', 'num_sequencia_tcp_destino',
       'tamanho_janela_tcp_destino', 'rtt_handshake_tcp', 'tempo_syn_synack',
       'tempo_synack_ack', 'tamanho_medio_pacotes_origem',
       'tamanho_medio_pacotes_destino', 'profundidade_transacao_http',
       'tamanho_corpo_resposta_http', 'conexoes_recentes_ip_porta_destino',
       'conexoes_recentes_ip_porta_origem', 'ftp_autenticada',
       'num_comandos_ftp', 'num_metodos_http',
       'ip_porta_

In [17]:
X_teste_reduzido_df = pd.DataFrame(X_teste_reduzido, columns=colunas_selecionadas)
X_teste_reduzido_df.head()

,total_bytes_origem,total_bytes_destino,carga_destino,pacotes_perdidos_origem,pacotes_perdidos_destino,rtt_handshake_tcp,tempo_syn_synack,tempo_synack_ack,tamanho_medio_pacotes_origem,tamanho_medio_pacotes_destino,protocolo_rede_outros,protocolo_rede_udp,servico_conexao_-,servico_conexao_ftp-data,estado_conexao_CON
0,-0.230576,-0.204932,-0.016329,-0.4,-0.333333,-0.009828,-0.016261,-0.004038,2.260274,-0.592593,0.0,1.0,1.0,0.0,0.0
1,0.298246,-0.204932,-0.016329,-0.4,-0.333333,-0.009828,-0.016261,-0.004038,10.931507,-0.592593,0.0,1.0,1.0,0.0,0.0
2,0.008354,-0.204932,-0.016329,-0.4,-0.333333,-0.009828,-0.016261,-0.004038,6.178082,-0.592593,0.0,1.0,1.0,0.0,0.0
3,-0.061821,-0.204932,-0.016329,-0.4,-0.333333,-0.009828,-0.016261,-0.004038,5.027397,-0.592593,0.0,1.0,1.0,0.0,0.0
4,0.450292,-0.204932,-0.016329,-0.4,-0.333333,-0.009828,-0.016261,-0.004038,13.424658,-0.592593,0.0,1.0,1.0,0.0,0.0


In [18]:
for feature in X_teste_reduzido_df.columns:
    print(f'\nFeature {feature.upper()}:')
    print(f'\n - Amplitude: de {X_teste_reduzido_df[feature].min()} a {X_teste_reduzido_df[feature].max()}')
    print(f'\n - Magnitude: {X_teste_reduzido_df[feature].max() - X_teste_reduzido_df[feature].min()}\n')


Feature TOTAL_BYTES_ORIGEM:

 - Amplitude: de -0.4277360066833751 a 5996.126148705096

 - Magnitude: 5996.55388471178


Feature TOTAL_BYTES_DESTINO:

 - Amplitude: de -0.20493247210804463 a 4303.239283617147

 - Magnitude: 4303.444216089255


Feature CARGA_DESTINO:

 - Amplitude: de -0.016328903360192524 a 34.078326484587734

 - Magnitude: 34.094655387947924


Feature PACOTES_PERDIDOS_ORIGEM:

 - Amplitude: de -0.4 a 1063.4

 - Magnitude: 1063.8000000000002


Feature PACOTES_PERDIDOS_DESTINO:

 - Amplitude: de -0.3333333333333333 a 917.5

 - Magnitude: 917.8333333333334


Feature RTT_HANDSHAKE_TCP:

 - Amplitude: de -0.009828122445103204 a 29.195324055790017

 - Magnitude: 29.20515217823512


Feature TEMPO_SYN_SYNACK:

 - Amplitude: de -0.016261070130149765 a 47.99016777600971

 - Magnitude: 48.00642884613986


Feature TEMPO_SYNACK_ACK:

 - Amplitude: de -0.00403792688985175 a 47.875280679371194

 - Magnitude: 47.879318606261045


Feature TAMANHO_MEDIO_PACOTES_ORIGEM:

 - Amplitude: d

#### 3.4. Normalização das features pós RFE
---

In [19]:
escalonador_robusto = joblib.load('../../analise_exploratoria_pre_processamento/escalonadores/escalonador_robusto.pkl')

In [20]:
features_normalizar = ['total_bytes_destino', 'carga_destino', 'pacotes_perdidos_destino', 'tempo_synack_ack', 'tamanho_medio_pacotes_origem']

##### 3.4.1. Features de treino
---

In [21]:
# Transforma o bloco de colunas inteiro de uma vez para respeitar o formato esperado pelo Scikit-Learn e evitar erros de dimensionalidade no .transform()
df_treino_rfe_normalizado = pd.DataFrame(
    escalonador_robusto.fit_transform(X_treino_reduzido_df[features_normalizar]), 
    columns=features_normalizar, 
    index=X_treino_reduzido_df.index
)

# Iteração por todos os atributos, mostrando os valores antes e depois da Normalização e apenas transformando os dados que já foram aprendidos
for col in features_normalizar:

    _, valor_min, valor_max, _, _, faixa = pm.gerar_dispersao(X_treino_reduzido_df, col)
    print(f'\n\n   -> Atributo {col.upper()}:\n')
    print(f'O atributo {col} possui amplitude entre {valor_min} e {valor_max} e magnitude {faixa} antes da Normalização com {escalonador_robusto}.\n')
    
    
    X_treino_reduzido_df.loc[:, col] = df_treino_rfe_normalizado[col]
    
    _, valor_min, valor_max, _, _, faixa = pm.gerar_dispersao(X_treino_reduzido_df, col)
    print(f'O atributo {col} foi normalizado. Sua amplitude atual está entre {valor_min} e {valor_max}. Já sua magnitude é {faixa}.\n')

# Consolida as alterações no DataFrame e previne o aviso "PerformanceWarning: DataFrame is highly fragmented"
X_treino_reduzido_df = X_treino_reduzido_df.copy()



   -> Atributo TOTAL_BYTES_DESTINO:

O atributo total_bytes_destino possui amplitude entre -0.20493247210804463 e 4302.6576629477395 e magnitude 4302.862595419848 antes da Normalização com RobustScaler().

O atributo total_bytes_destino foi normalizado. Sua amplitude atual está entre -0.20493247210804463 e 4302.6576629477395. Já sua magnitude é 4302.862595419848.



   -> Atributo CARGA_DESTINO:

O atributo carga_destino possui amplitude entre -0.016328903360192524 e 36.700989512499405 e magnitude 36.717318415859594 antes da Normalização com RobustScaler().

O atributo carga_destino foi normalizado. Sua amplitude atual está entre -0.016328903360192524 e 36.700989512499405. Já sua magnitude é 36.717318415859594.



   -> Atributo PACOTES_PERDIDOS_DESTINO:

O atributo pacotes_perdidos_destino possui amplitude entre -0.3333333333333333 e 913.6666666666666 e magnitude 914.0 antes da Normalização com RobustScaler().

O atributo pacotes_perdidos_destino foi normalizado. Sua amplitude atual

##### 3.4.2. Features de teste
---

In [22]:
# Transforma o bloco de colunas inteiro de uma vez para respeitar o formato esperado pelo Scikit-Learn e evitar erros de dimensionalidade no .transform()
df_teste_rfe_normalizado = pd.DataFrame(
    escalonador_robusto.transform(X_teste_reduzido_df[features_normalizar]), 
    columns=features_normalizar, 
    index=X_teste_reduzido_df.index
)

# Iteração por todos os atributos, mostrando os valores antes e depois da Normalização e apenas transformando os dados que já foram aprendidos
for col in features_normalizar:

    _, valor_min, valor_max, _, _, faixa = pm.gerar_dispersao(X_teste_reduzido_df, col)
    print(f'\n\n   -> Atributo {col.upper()}:\n')
    print(f'O atributo {col} possui amplitude entre {valor_min} e {valor_max} e magnitude {faixa} antes da Normalização com {escalonador_robusto}.\n')
    
    
    X_teste_reduzido_df.loc[:, col] = df_teste_rfe_normalizado[col]
    
    _, valor_min, valor_max, _, _, faixa = pm.gerar_dispersao(X_teste_reduzido_df, col)
    print(f'O atributo {col} foi normalizado. Sua amplitude atual está entre {valor_min} e {valor_max}. Já sua magnitude é {faixa}.\n')

# Consolida as alterações no DataFrame e previne o aviso "PerformanceWarning: DataFrame is highly fragmented"
X_teste_reduzido_df = X_teste_reduzido_df.copy()



   -> Atributo TOTAL_BYTES_DESTINO:

O atributo total_bytes_destino possui amplitude entre -0.20493247210804463 e 4303.239283617147 e magnitude 4303.444216089255 antes da Normalização com RobustScaler().

O atributo total_bytes_destino foi normalizado. Sua amplitude atual está entre -0.20493247210804463 e 4303.239283617147. Já sua magnitude é 4303.444216089255.



   -> Atributo CARGA_DESTINO:

O atributo carga_destino possui amplitude entre -0.016328903360192524 e 34.078326484587734 e magnitude 34.094655387947924 antes da Normalização com RobustScaler().

O atributo carga_destino foi normalizado. Sua amplitude atual está entre -0.016328903360192524 e 34.078326484587734. Já sua magnitude é 34.094655387947924.



   -> Atributo PACOTES_PERDIDOS_DESTINO:

O atributo pacotes_perdidos_destino possui amplitude entre -0.3333333333333333 e 917.5 e magnitude 917.8333333333334 antes da Normalização com RobustScaler().

O atributo pacotes_perdidos_destino foi normalizado. Sua amplitude atual e

## D) Desenvolvimento dos Modelos de MLP de Classificação de duas camadas sem a aplicação do RFE
---

## D.1.) Primeiro modelo - Arquitetura em funil descrecente, sem RFE
---

### 1. Treino do modelo
---

#### 1.1. Instanciando a classe MLPClassifier
---

In [23]:
modelo_1 = MLPClassifier(hidden_layer_sizes=(256,128,64), max_iter=2000, tol=0.0001, learning_rate_init=0.001, learning_rate='adaptive', validation_fraction=0.1, alpha=0.001, n_iter_no_change=15, solver="adam", activation='relu', verbose=True, random_state=42)

#### 1.2. Explicando os hiperparâmetros utilizados em MLPClassifier:
---

- `hidden_layer_sizes=(64,32)`: quantidade de camadas ocultas e quantos neurônios há em cada uma. No caso, a rede possui duas camadas ocultas: a primeira com 64 neurônios e a segunda com 32.
- `max_iter=2000`: limite máximo de iterações de treinamento. Se chegar a 2000 voltas, o treinamento para obrigatoriamente, mesmo que não tenha atingido a perfeição.
- `tol=0.0001`: tolerância para melhoria do erro (loss). Se a diferença de erro entre uma iteração e outra for menor do que 0.0001, o Python entende que a rede estabilizou e parou de evoluir significativamente.
- `learning_rate_init=0.001`: taxa de aprendizado inicial (tamanho do passo). Define o quão grandes serão os passos que o otimizador dará para ajustar os pesos. Passos muito grandes (0.1) podem fazer a rede neural ignorar a solução ideal; passos muito pequenos (0.00001) fazem o treinamento demorar muito mais. O 0.001 é um bom meio-termo, trazendo eficiência e um menor tempo de aprendizagem. A fórmula utilizada pelo modelo do Scikit-Learn é a `Gradiente Descendente`: $$w_{novo} = w_{antigo} - \eta \cdot \frac{\partial Loss}{\partial w}$$
- `learning_rate='adaptive'`: comportamento da taxa de aprendizado ao longo do tempo. Enquanto o modelo estiver conseguindo diminuir o erro, ele mantém o passo inicial (0.001). Se o erro parar de cair por duas épocas seguidas, a rede divide a taxa de aprendizado por 5, dando passos menores e mais cautelosos para tentar melhorar.
- `validation_fraction=0.1`: Reserva 10% dos seus dados de treino para usar como validação. A rede treina nos outros 90% e se testa nesses 10% para saber se está realmente aprendendo ou apenas decorando. É com base nessa nota que o `n_iter_no_change` e o `learning_rate='adaptive'` tomam suas decisões.
- `alpha=0.001`: evita o `overfitting` (regularização L2). Penaliza pesos que fiquem grandes demais na rede. Isso força o modelo a se manter simples e impede que ele decore o banco de dados de treino, ajudando-o a funcionar bem com dados novos de verdade. A fórmula de Regularização L2 (ou Ridge) é: $$Loss_{total} = Loss_{original} + \alpha \sum_{i} w_i^2$$
- `n_iter_no_change=15`: número de iterações que a rede vai esperar se a melhora do erro ficar abaixo daquela tolerância (tol). Se por 15 vezes seguidas o modelo não melhorar quase nada, ele aciona o "early stopping" (parada precoce) e encerra o treino para economizar tempo.
- `solver="adam"`: motor matemático que ajusta os pesos da rede otimizando-a. O `adam` é o algoritmo padrão do mercado. Funciona como um piloto automático inteligente que descobre sozinho o melhor caminho para diminuir os erros do modelo de forma rápida e eficiente. A fórmula simplificada de como o adam ajusta o passo a cada iteração $t$ é:$$w_t = w_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$
- `activation='relu'`: função de ativação dos neurônios. Decide se o neurônio deve ser ativado ou não. `A ReLU` (Rectified Linear Unit) pega qualquer número negativo e transforma em zero, e mantém os números positivos como estão. É a mais usada porque ajuda a rede a aprender padrões complexos rapidamente sem travar o código. Sua fórmula é:
  $f(x) = \max(0, x)$
- `verbose=True`: exibe na tela do terminal o progresso do erro a cada iteração que passa. Essencial para acompanhar se o modelo está evoluindo ou se travou.
- `random_state=42`: semente de aleatoriedade. Redes neurais começam com pesos aleatórios e embaralham os dados antes de treinar. Definir o número 42 (ou qualquer outro número fixo) garante que a aleatoriedade seja exatamente a mesma toda vez que  rodar o script. Assim, os resultados serão iguais e replicáveis.

#### 1.3. Treino do modelo
---

Para o treino do modelo, é usada a expressão `modelo.fit(X_treino, y_treino)`. Será rodado em conjunto com a função `imprimir_desempenho()` do módulo `metricas` do pacote `Predexan`, para poder ter o tempo de processamento registrado: 

In [ ]:
pm.imprimir_desempenho(modelo_1.fit,X_treino, y_treino)

Iteration 1, loss = 0.38896686
Iteration 2, loss = 0.24650777
Iteration 3, loss = 0.21104215
Iteration 4, loss = 0.20148910
Iteration 5, loss = 0.19481977
Iteration 6, loss = 0.19113667
Iteration 7, loss = 0.18894383
Iteration 8, loss = 0.18779286
Iteration 9, loss = 0.18982011
Iteration 10, loss = 0.18649699
Iteration 11, loss = 0.18483810
Iteration 12, loss = 0.18396279
Iteration 13, loss = 0.18315009
Iteration 14, loss = 0.18359019
Iteration 15, loss = 0.18176738
Iteration 16, loss = 0.18266987
Iteration 17, loss = 0.18342066
Iteration 18, loss = 0.17935247
Iteration 19, loss = 0.17911303
Iteration 20, loss = 0.17930183
Iteration 21, loss = 0.17825141
Iteration 22, loss = 0.17880347
Iteration 23, loss = 0.17839045
Iteration 24, loss = 0.17753854
Iteration 25, loss = 0.17591160
Iteration 26, loss = 0.17538775
Iteration 27, loss = 0.17552865
Iteration 28, loss = 0.17454083
Iteration 29, loss = 0.17382115
Iteration 30, loss = 0.17340126
Iteration 31, loss = 0.17343562
Iteration 32, los

### 2. Realização de predições com os dados de teste
---

Para a predição probabilística, é usado o método `predict_proba()` na seguinte expressão: `modelo.predict_proba(X_teste)`. Para a predição binária, é expresso da mesma forma, mas com o uso do método `predict()`. Ambas terão seu desempenho medido pela função `imprimir_desempenho()` de `Predexan`.

#### 2.1. Predição Probababilística
---

Para a Predição Probabilística, é usado o método `predict_proba()` na seguinte expressão: `modelo.predict_proba(X_teste)`

In [ ]:
y_predicao_probabilistica_1 = pm.imprimir_desempenho(modelo_1.predict_proba, X_teste)[:,1]

#### 2.2. Predição Binária
---

Para a Predição Binária, é usado o método `predict()` na seguinte expressão: `modelo.predict(X_teste)`

In [ ]:
y_predicao_binaria_1 = pm.imprimir_desempenho(modelo_1.predict, X_teste)

In [ ]:
type(y_predicao_binaria_1)

### 3. Desempenho do modelo
---

O modelo possui métricas de desempenho baseadas nos resultados dos testes, sendo essas: 

- `Pontuação` que traz a `Acurácia Geral`.
- `Matriz de Confusão`:  permite a visualização do desempenho do modelo, uma vez que mostra o total de predições Verdadeiras Positivas (TP), Verdadeiras Negativa (TN), Falsas Positivas (FP) e Falsas Negativas (FN). No contexto do projeto, que visa a probailidade de detecção de ameaças, os alvos são sempre atingir ao máximoTN e TP, mas FN é o erro crítico, pois pode levar ao usuário da aplicação a ignorar a hipótese do ataque em sua máquina, podendo propiciar uma piora no caso, já que este pode deixar de buscar ajuda e manutenções e FP é o alarme falso.
- `Relatório de Classificação` que traz cálculos como `Precision`, `Recall` e `F1-Score`.
- `Curva ROC AUC` que traz uma visualização gráfica.
- `Curva Precision-Recall`

Serão feitos testes com limiares para se determinar qual deles tem melhor desempenho, revocação e precisão. As seguintes sub-seções trarão as métricas e gráficos de forma individual para cada teste.

#### 3.1. Limiar Padrão (0.5)
---

##### 3.1.1. Matriz de Confusão
---

In [ ]:
matriz_confusao_1 = confusion_matrix(y_teste, y_predicao_binaria_1)
sns.heatmap(matriz_confusao_1, annot=True, fmt='d', cmap='Blues')

##### 3.1.2. Relatório de Classificação
---

In [ ]:
relatorio_classificacao_1 = classification_report(y_teste, y_predicao_binaria_1)
print(relatorio_classificacao_1)

##### 3.1.3. Curva ROC-AUC
---

In [ ]:
pg.gerar_roc_auc(modelo_1, X_teste,y_teste)

##### 3.1.4. Curva Precision-Recall
---

In [ ]:
pg.gerar_curva_pr(y_teste, y_predicao_probabilistica_1)

In [ ]:
pm.imprimir_prt(y_teste, y_predicao_probabilistica_1, obj_recall=True, meta_recall=0.95, obj_precision=True, meta_precision=0.95, obj_f1=True)

#### 3.2. Limiar Equilibrado
---

##### 3.2.1. Matriz de Confusão
---

Limiar de equilíbrio do F1-Score entre Recall e Precision:

In [ ]:
_, limiar_modelo_1, _ = pm.extrair_f1(y_teste, y_predicao_probabilistica_1)
print(limiar_modelo_1)

In [ ]:
y_predicao_probabilistica_equilibrada_1 = (y_predicao_probabilistica_1 > limiar_modelo_1).astype(int)

In [ ]:
matriz_confusao_equilibrada_1 = confusion_matrix(y_teste, y_predicao_probabilistica_equilibrada_1)
sns.heatmap(matriz_confusao_equilibrada_1, annot=True, fmt='d', cmap='BrBG')

##### 3.2.2. Relatório de Classificação
---

In [ ]:
relatorio_classificacao_equilibrado_1 = classification_report(y_teste, y_predicao_probabilistica_equilibrada_1)
print(relatorio_classificacao_equilibrado_1)

## D.2.) Segundo modelo - Arquitetura em funil crescente, sem RFE
---

### 1. Treino do modelo
---

#### 1.1. Instanciando a classe MLPClassifier
---

In [ ]:
modelo_2 = MLPClassifier(hidden_layer_sizes=(64,128,256), max_iter=2000, tol=0.0001, learning_rate_init=0.001, learning_rate='adaptive', validation_fraction=0.1, alpha=0.001, n_iter_no_change=15, solver="adam", activation='relu', verbose=True, random_state=42)

#### 1.2. Treino do modelo
---

In [ ]:
pm.imprimir_desempenho(modelo_2.fit,X_treino, y_treino)

### 2. Realização de predições com os dados de teste
---

#### 2.1. Predição Probababilística
---

In [ ]:
y_predicao_probabilistica_2 = pm.imprimir_desempenho(modelo_2.predict_proba, X_teste)[:,1]

#### 2.2. Predição Binária
---

In [ ]:
y_predicao_binaria_2 = pm.imprimir_desempenho(modelo_2.predict, X_teste)

### 3. Desempenho do modelo
---

#### 3.1. Limiar Padrão (0.5)
---

##### 3.1.1. Matriz de Confusão
---

In [ ]:
matriz_confusao_2 = confusion_matrix(y_teste, y_predicao_binaria_2)
sns.heatmap(matriz_confusao_2, annot=True, fmt='d', cmap='Purples')

In [ ]:
pm.imprimir_desempenho_modelo(matriz_confusao_2)

##### 3.1.2. Relatório de Classificação
---

In [ ]:
relatorio_classificacao_2 = classification_report(y_teste, y_predicao_binaria_2)
print(relatorio_classificacao_2)

##### 3.1.3. Curva ROC-AUC
---

In [ ]:
pg.gerar_roc_auc(modelo_2, X_teste,y_teste)

##### 3.1.4. Curva Precision-Recall
---

In [ ]:
pg.gerar_curva_pr(y_teste, y_predicao_probabilistica_2)

In [ ]:
pm.imprimir_prt(y_teste, y_predicao_probabilistica_2, obj_recall=True, meta_recall=0.95, obj_precision=True, meta_precision=0.95, obj_f1=True)

#### 3.2. Limiar Equilibrado
---

##### 3.2.1. Matriz de Confusão
---

In [ ]:
_, limiar_modelo_2, _ = pm.extrair_f1(y_teste, y_predicao_probabilistica_2)

In [ ]:
y_predicao_probabilistica_equilibrada_2 = (y_predicao_probabilistica_2 > limiar_modelo_2).astype(int)

In [ ]:
matriz_confusao_equilibrada_2 = confusion_matrix(y_teste, y_predicao_probabilistica_equilibrada_2)
sns.heatmap(matriz_confusao_equilibrada_2, annot=True, fmt='d', cmap='magma')

In [ ]:
pm.imprimir_desempenho_modelo(matriz_confusao_equilibrada_2)

##### 3.2.2. Relatório de Classificação
---

In [ ]:
relatorio_classificacao_equilibrado_2 = classification_report(y_teste, y_predicao_probabilistica_equilibrada_2)
print(relatorio_classificacao_equilibrado_2)

## E) Desenvolvimento dos Modelo de MLPs de Classificação de duas camadas com 15 features (RFE + XGBoost Classifier)
---

## E.1.) Terceiro modelo - Arquitetura em funil descrescente, com RFE
---

### 1. Treino do modelo
---

#### 1.1. Instanciando a classe MLPClassifier
---

In [ ]:
modelo_3 = MLPClassifier(hidden_layer_sizes=(256,128,64), max_iter=2000, tol=0.0001, learning_rate_init=0.001, learning_rate='adaptive', validation_fraction=0.1, alpha=0.001, n_iter_no_change=15, solver="adam", activation='relu', verbose=True, random_state=42)

#### 1.2. Treino do modelo
---

In [ ]:
pm.imprimir_desempenho(modelo_3.fit,X_treino_reduzido_df, y_treino)

### 2. Realização de predições com os dados de teste
---

#### 2.1. Predição Probababilística
---

In [ ]:
y_predicao_probabilistica_3 = pm.imprimir_desempenho(modelo_3.predict_proba, X_teste_reduzido_df)[:,1]

#### 2.2. Predição Binária
---

In [ ]:
y_predicao_binaria_3 = pm.imprimir_desempenho(modelo_3.predict, X_teste_reduzido_df)

### 3. Desempenho do modelo
---

#### 3.1. Limiar Padrão (0.5)
---

##### 3.1.1 Matriz de Confusão
---

In [ ]:
matriz_confusao_3 = confusion_matrix(y_teste, y_predicao_binaria_3)
sns.heatmap(matriz_confusao_3, annot=True, fmt='d', cmap='Greens')

In [ ]:
pm.imprimir_desempenho_modelo(matriz_confusao_3)

##### 3.1.2. Relatório de Classificação
---

In [ ]:
relatorio_classificacao_3 = classification_report(y_teste, y_predicao_binaria_3)
print(relatorio_classificacao_3)

##### 3.1.3. Curva ROC-AUC
---

In [ ]:
pg.gerar_roc_auc(modelo_3, X_teste_reduzido_df,y_teste)

##### 3.1.4. Curva Precision-Recall
---

In [ ]:
pg.gerar_curva_pr(y_teste, y_predicao_probabilistica_3)

In [ ]:
pm.imprimir_prt(y_teste, y_predicao_probabilistica_3, obj_recall=True, meta_recall=0.95, obj_precision=True, meta_precision=0.95, obj_f1=True, obj_f2=True)

#### 3.2. Limiar Equilibrado
---

##### 3.2.1 Matriz de Confusão
---

In [ ]:
_, limiar_modelo_3, _ = pm.extrair_f1(y_teste, y_predicao_probabilistica_3)

In [ ]:
y_predicao_probabilistica_equilibrada_3 = (y_predicao_probabilistica_3 > limiar_modelo_3).astype(int)

In [ ]:
matriz_confusao_equilibrada_3 = confusion_matrix(y_teste, y_predicao_probabilistica_equilibrada_3)
sns.heatmap(matriz_confusao_equilibrada_3, annot=True, fmt='d', cmap='plasma')

In [ ]:
pm.imprimir_desempenho_modelo(matriz_confusao_equilibrada_3)

##### 3.2.2. Relatório de Classificação
---

In [ ]:
relatorio_classificacao_equilibrado_3 = classification_report(y_teste, y_predicao_probabilistica_equilibrada_3)
print(relatorio_classificacao_equilibrado_3)

## E.2.) Quarto modelo - Arquitetura em funil Crescente, com RFE
---

### 1. Treino do modelo
---

#### 1.1. Instanciando a classe MLPClassifier
---

In [ ]:
modelo_4 = MLPClassifier(hidden_layer_sizes=(64,128,256), max_iter=2000, tol=0.0001, learning_rate_init=0.001, learning_rate='adaptive', validation_fraction=0.1, alpha=0.001, n_iter_no_change=15, solver="adam", activation='relu', verbose=True, random_state=42)

#### 1.2. Treino do modelo
---

In [ ]:
pm.imprimir_desempenho(modelo_4.fit,X_treino_reduzido_df, y_treino)

### 2. Realização de predições com os dados de teste
---

#### 2.1. Predição Probababilística
---

In [ ]:
y_predicao_probabilistica_4 = pm.imprimir_desempenho(modelo_4.predict_proba, X_teste_reduzido_df)[:,1]

#### 2.2. Predição Binária
---

In [ ]:
y_predicao_binaria_4 = pm.imprimir_desempenho(modelo_4.predict, X_teste_reduzido_df)

### 3. Desempenho do modelo
---

#### 3.1. Limiar Padrão (0.5)
---

##### 3.1.1 Matriz de Confusão
---

In [ ]:
matriz_confusao_4 = confusion_matrix(y_teste, y_predicao_binaria_4)
sns.heatmap(matriz_confusao_4, annot=True, fmt='d', cmap='YlOrRd')

In [ ]:
pm.imprimir_desempenho_modelo(matriz_confusao_4)

##### 3.1.2. Relatório de Classificação
---

In [ ]:
relatorio_classificacao_4 = classification_report(y_teste, y_predicao_binaria_4)
print(relatorio_classificacao_3)

##### 3.1.3. Curva ROC-AUC
---

In [ ]:
pg.gerar_roc_auc(modelo_4, X_teste_reduzido_df,y_teste)

##### 3.1.4. Curva Precision-Recall
---

In [ ]:
pg.gerar_curva_pr(y_teste, y_predicao_probabilistica_4)

#### 3.2. Limiar Equilibrado
---

##### 3.2.1 Matriz de Confusão
---

In [ ]:
_, limiar_modelo_4, _ = pm.extrair_f1(y_teste, y_predicao_probabilistica_4)

In [ ]:
y_predicao_probabilistica_equilibrada_4 = (y_predicao_probabilistica_4 > limiar_modelo_4).astype(int)

In [ ]:
matriz_confusao_equilibrada_4 = confusion_matrix(y_teste, y_predicao_probabilistica_equilibrada_4)
sns.heatmap(matriz_confusao_equilibrada_4, annot=True, fmt='d', cmap='vlag')

In [ ]:
pm.imprimir_desempenho_modelo(matriz_confusao_equilibrada_4)

##### 3.2.2. Relatório de Classificação
---

In [ ]:
relatorio_classificacao_equilibrado_4 = classification_report(y_teste, y_predicao_binaria_4)
print(relatorio_classificacao_equilibrado_4)

## F) Comparando os Modelos
---

In [ ]:
print('\nComparação entre os modelos à partir de Precision, Recall e F1-Score da classe 1:\n')

print('\n -> Modelo 1:')
print('\n - Padrão:\n')
pm.imprimir_desempenho_modelo(matriz_confusao_1)
print('\n - Equilibrado:\n')
pm.imprimir_desempenho_modelo(matriz_confusao_equilibrada_1)

print('\n\n -> Modelo 2:')
print('\n - Padrão:\n')
pm.imprimir_desempenho_modelo(matriz_confusao_2)
print('\n - Equilibrado:\n')
pm.imprimir_desempenho_modelo(matriz_confusao_equilibrada_2)

print('\n\n -> Modelo 3:')
print('\n - Padrão:\n')
pm.imprimir_desempenho_modelo(matriz_confusao_3)
print('\n - Equilibrado:\n')
pm.imprimir_desempenho_modelo(matriz_confusao_equilibrada_3)

print('\n\n -> Modelo 4:')
print('\n - Padrão:\n')
pm.imprimir_desempenho_modelo(matriz_confusao_4)
print('\n - Equilibrado:\n')
pm.imprimir_desempenho_modelo(matriz_confusao_equilibrada_4)

## G) Persisitindo os Modelos
---

### 1. Primeiro modelo - Arquitetura em Funil Decrescente, sem RFE
---

In [ ]:
# joblib.dump(modelo_1, '../modelos_persistentes/mlp/mlp_3c/mlp_3c_decrescente.pkl')

### 2. Segundo modelo - Arquitetura em Funil Crescente, sem RFE
---

In [ ]:
# joblib.dump(modelo_2, '../modelos_persistentes/mlp/mlp_3c/mlp_rfe_3c_crescente.pkl')

### 3. Terceiro modelo - Arquitetura em Funil Decrescente, com RFE
---

In [ ]:
# joblib.dump(modelo_3, '../modelos_persistentes/mlp/mlp_3c/mlp_3c_invertido_decrescente.pkl')

### 4. Quarto modelo - Arquitetura em Funil Crescente, com RFE
---

In [ ]:
# joblib.dump(modelo_4, '../modelos_persistentes/mlp/mlp_3c/mlp_rfe_3c_invertido_crescente.pkl')